# Reproducibility notebook — *A Phase-Space Geometric Measure of Magic in Qubit Systems*

S. Dutta and Tushar, arXiv:2603.20792 (v4).

This notebook recomputes every numerical claim in the paper and checks every
closed-form result against linear programming. Each check is recorded in a ledger
that is printed at the end (Section 13).

**Conventions** (as in the paper)
* Single-qubit Wootters phase-point operators
  $A_{(q,p)} = \tfrac12\bigl(I + (-1)^p X + (-1)^{q+p} Y + (-1)^q Z\bigr)$,
  $n$-qubit operators are tensor products, and $W_\rho(\alpha)=\mathrm{Tr}(\rho A_\alpha)/2^n$.
* $C(\rho)=\min_{\sigma\in\mathrm{conv}(\mathrm{STAB}_n)}\lVert W_\rho-W_\sigma\rVert_1$ (linear program).
* $\Gamma(\rho)$ is the robustness of magic, $\kappa=(\Gamma-1)/C$.
* Logical operators of the $[[n,1,1]]$ repetition code: $X_L=X^{\otimes n}$,
  $Y_L=Y\otimes X^{\otimes(n-1)}$, $Z_L=Z\otimes I^{\otimes(n-1)}$.

**Runtime.** With `RUN_N4 = RUN_SLOW = False` the notebook runs in a few minutes.
`RUN_N4` adds the four-qubit linear programs (36,720 stabilizer states; roughly
10–20 minutes). `RUN_SLOW` adds the frame-census maxima and multistart searches
(roughly 15 minutes).

**Requirements.** Python ≥ 3.9, `numpy`, `scipy` (HiGHS solver, ≥ 1.9).

In [1]:
RUN_N4   = True    # four-qubit LPs (Section 10)
RUN_SLOW = True    # multistart searches and the full frame census maxima (Sections 4, 11)

import numpy as np, itertools, time, collections
from scipy.optimize import linprog, minimize
from scipy import sparse
np.set_printoptions(precision=6, suppress=True, linewidth=110)
T0 = time.time()

LEDGER = []
def check(name, ok, detail=""):
    """Record a check in the ledger and print it."""
    ok = bool(ok)
    LEDGER.append((name, ok, detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  —  {detail}" if detail else ""))

def close(a, b, tol=1e-7):
    return abs(a - b) <= tol

## 1. Machinery: phase-point operators, stabilizer states, and the two linear programs

In [2]:
I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z  = np.diag([1, -1]).astype(complex)
Hd = (X + Z) / np.sqrt(2)
Sg = np.diag([1, 1j]).astype(complex)
P1 = [I2, X, Y, Z]

def kron_all(ops):
    out = np.array([[1]], dtype=complex)
    for o in ops:
        out = np.kron(out, o)
    return out

def A1q(q, p, a=(1, 1, 1)):
    """Single-qubit phase-point operator; a=(a_x,a_y,a_z) selects one of the 8 valid frames."""
    return 0.5 * (I2 + a[0] * (-1)**p * X + a[1] * (-1)**(q + p) * Y + a[2] * (-1)**q * Z)

A1 = [A1q(q, p) for q in (0, 1) for p in (0, 1)]        # order (q,p) = 00, 01, 10, 11

def phase_points(n, single=None):
    single = A1 if single is None else single
    return [kron_all(t) for t in itertools.product(single, repeat=n)]

def wigner_matrix(Aops):
    """Matrix M with W = Re(M @ vec(rho)) / dim."""
    return np.array([A.T.reshape(-1) for A in Aops])

def make_wigner(n, Aops=None):
    Aops = phase_points(n) if Aops is None else Aops
    M = wigner_matrix(Aops); dim = 2**n
    return lambda rho: (M @ np.asarray(rho).reshape(-1)).real / dim

def proj(v):
    v = np.asarray(v, dtype=complex); v = v / np.linalg.norm(v)
    return np.outer(v, v.conj())

def bloch(r):
    return 0.5 * (I2 + r[0] * X + r[1] * Y + r[2] * Z)

def stabilizer_states(n):
    """All pure n-qubit stabilizer states, as the orbit of |0..0> under H, S and CNOT."""
    dim = 2**n
    gates = []
    for i in range(n):
        for g in (Hd, Sg):
            ops = [I2] * n; ops[i] = g; gates.append(kron_all(ops))
    for c, t in itertools.permutations(range(n), 2):
        M = np.zeros((dim, dim), dtype=complex)
        for b in range(dim):
            bits = [(b >> (n - 1 - k)) & 1 for k in range(n)]
            if bits[c]: bits[t] ^= 1
            M[int(''.join(map(str, bits)), 2), b] = 1
        gates.append(M)
    def key(v):
        i = np.flatnonzero(np.abs(v) > 1e-9)[0]
        v = v * (abs(v[i]) / v[i])
        return tuple(np.round(np.concatenate([v.real, v.imag]), 7))
    start = np.zeros(dim, dtype=complex); start[0] = 1
    seen = {key(start): start}; frontier = [start]
    while frontier:
        nxt = []
        for v in frontier:
            for g in gates:
                w = g @ v; k = key(w)
                if k not in seen:
                    seen[k] = w; nxt.append(w)
        frontier = nxt
    return np.array(list(seen.values()))

class Magic:
    """C and Gamma for n qubits in a given frame (default: Wootters product frame)."""
    def __init__(self, n, Aops=None, states=None):
        self.n = n; self.dim = 2**n
        self.Aops = phase_points(n) if Aops is None else Aops
        self.M = wigner_matrix(self.Aops)
        self.psi = stabilizer_states(n) if states is None else states
        R = np.einsum('si,sj->sij', self.psi, self.psi.conj()).reshape(len(self.psi), -1)
        self.Ws = (R @ self.M.T).real / self.dim                 # (N_stab, 4^n)
        self.N, self.d = self.Ws.shape
        self._buildC(); self._buildG()
    def W(self, rho):
        return (self.M @ np.asarray(rho).reshape(-1)).real / self.dim
    def _buildC(self):
        N, d = self.N, self.d
        Wt = sparse.csr_matrix(self.Ws.T)
        Id = sparse.identity(d, format='csr')
        self.cC = np.r_[np.zeros(N), np.ones(d)]
        self.AubC = sparse.vstack([sparse.hstack([-Wt, -Id]), sparse.hstack([Wt, -Id])]).tocsr()
        self.AeqC = sparse.csr_matrix(np.r_[np.ones(N), np.zeros(d)][None, :])
    def _buildG(self):
        Wt = sparse.csr_matrix(self.Ws.T)
        self.AeqG = sparse.hstack([Wt, -Wt]).tocsr()
    def C(self, rho, witness=False):
        w0 = self.W(rho)
        r = linprog(self.cC, A_ub=self.AubC, b_ub=np.r_[-w0, w0], A_eq=self.AeqC, b_eq=[1],
                    bounds=(0, None), method='highs')
        assert r.status == 0, r.message
        if not witness:
            return r.fun
        m = r.ineqlin.marginals; y = m[:self.d] - m[self.d:]
        for cand in (y, -y):
            if close(cand @ w0 - (self.Ws @ cand).max(), r.fun, 1e-6):
                return r.fun, cand
        return r.fun, None
    def Gamma(self, rho):
        w0 = self.W(rho)
        r = linprog(np.ones(2 * self.N), A_eq=self.AeqG, b_eq=w0, bounds=(0, None), method='highs')
        assert r.status == 0, r.message
        return r.fun
    def M_n(self):
        return np.abs(self.Ws).sum(1).max()

expected_counts = {1: 6, 2: 60, 3: 1080, 4: 36720}
MAG = {}
for n in (1, 2, 3):
    t = time.time(); MAG[n] = Magic(n)
    check(f"number of {n}-qubit stabilizer states = {expected_counts[n]}",
          MAG[n].N == expected_counts[n], f"found {MAG[n].N} ({time.time()-t:.1f}s)")
if RUN_N4:
    t = time.time(); MAG[4] = Magic(4)
    check("number of 4-qubit stabilizer states = 36720", MAG[4].N == 36720,
          f"found {MAG[4].N} ({time.time()-t:.1f}s)")
M1, M2 = MAG[1], MAG[2]
W1, W2 = M1.W, M2.W

[PASS] number of 1-qubit stabilizer states = 6  —  found 6 (0.0s)
[PASS] number of 2-qubit stabilizer states = 60  —  found 60 (0.0s)
[PASS] number of 3-qubit stabilizer states = 1080  —  found 1080 (0.2s)


[PASS] number of 4-qubit stabilizer states = 36720  —  found 36720 (15.6s)


### 1.1 Basic Wigner facts (Section 2.3, Appendix C)

In [3]:
check("M_1 = 1", close(M1.M_n(), 1))
check("M_2 = 2 (attained by Bell states)", close(M2.M_n(), 2))
print(f"   for reference: M_3 = {MAG[3].M_n():.6f}" + (f", M_4 = {MAG[4].M_n():.6f}" if RUN_N4 else ""))
rng = np.random.default_rng(2603)
def rand_pure(dim):
    v = rng.normal(size=dim) + 1j * rng.normal(size=dim); return proj(v)
def rand_mixed(dim, rank=None):
    rank = dim if rank is None else rank
    G = rng.normal(size=(dim, rank)) + 1j * rng.normal(size=(dim, rank)); r = G @ G.conj().T
    return r / np.trace(r).real
samples = [W2(rand_pure(4)) for _ in range(20000)]
wmin = min(w.min() for w in samples); wmax = max(w.max() for w in samples)
negcount = collections.Counter(int((w < -1e-12).sum()) for w in samples)
ev = np.linalg.eigvalsh(np.kron(A1[0], A1[0]))
check("eigenvalues of a two-qubit A_alpha are -1/2, -1/2, (2-√3)/2, (2+√3)/2",
      np.allclose(sorted(ev), sorted([-.5, -.5, (2-3**.5)/2, (2+3**.5)/2])))
check("Prop C.1(2): -1/8 <= W <= (2+√3)/8 on 20,000 random pure states",
      wmin >= -1/8 - 1e-12 and wmax <= (2+3**.5)/8 + 1e-12, f"observed range [{wmin:.4f}, {wmax:.4f}]")
check("Prop C.1(3): sum W^2 = 1/4 for pure states",
      all(close((w**2).sum(), .25, 1e-10) for w in samples[:500]))
print("   number of negative entries over random pure two-qubit states:", dict(sorted(negcount.items())))
check("Appendix C: random pure states reach 9 negative entries (the old 'at most 6' claim was false)",
      max(negcount) >= 9, f"maximum seen {max(negcount)}")

[PASS] M_1 = 1
[PASS] M_2 = 2 (attained by Bell states)
   for reference: M_3 = 2.500000, M_4 = 4.000000


[PASS] eigenvalues of a two-qubit A_alpha are -1/2, -1/2, (2-√3)/2, (2+√3)/2
[PASS] Prop C.1(2): -1/8 <= W <= (2+√3)/8 on 20,000 random pure states  —  observed range [-0.1249, 0.4584]
[PASS] Prop C.1(3): sum W^2 = 1/4 for pure states
   number of negative entries over random pure two-qubit states: {1: 1, 2: 15, 3: 219, 4: 3210, 5: 8213, 6: 6630, 7: 1489, 8: 200, 9: 23}
[PASS] Appendix C: random pure states reach 9 negative entries (the old 'at most 6' claim was false)  —  maximum seen 9


## 2. The single-qubit metric: cuboctahedron = $A_3$ root polytope (Lemma 4.1, Remark 4.2)

In [4]:
Tset = np.array([t for t in itertools.product([1, -1], repeat=3) if np.prod(t) == 1])
vs = rng.normal(size=(20000, 3))
lhs = np.abs(vs @ Tset.T).sum(1) / 4
rhs = np.maximum(np.abs(vs).max(1), 0.5 * np.abs(vs).sum(1))
check("Lemma 4.1: ||W_rho - W_sigma||_1 = max(||v||_inf, ||v||_1/2) (20,000 random v)",
      np.allclose(lhs, rhs, atol=1e-12))
# direct check with actual states
ok = True
for _ in range(500):
    r1, r2 = rng.normal(size=3), rng.normal(size=3)
    r1 /= max(1, np.linalg.norm(r1)); r2 /= max(1, np.linalg.norm(r2))
    v = r1 - r2
    ok &= close(np.abs(W1(bloch(r1)) - W1(bloch(r2))).sum(), max(np.abs(v).max(), .5*np.abs(v).sum()), 1e-12)
check("Lemma 4.1 with actual single-qubit Wigner functions (500 pairs)", ok)
# root polytope: the 12 vertices of K map to (e_a - e_b)/2
verts = [np.array(p) for p in set(itertools.permutations([1, 1, 0])) | set(itertools.permutations([1, -1, 0])) |
         set(itertools.permutations([-1, -1, 0])) | set(itertools.permutations([-1, 1, 0]))]
images = set()
for v in verts:
    dW = W1(bloch(np.zeros(3)) + 0.5 * (v[0]*X + v[1]*Y + v[2]*Z)) - W1(bloch(np.zeros(3)))
    images.add(tuple(np.round(dW * 2, 9)))
roots = {tuple(np.round(np.eye(4)[a] - np.eye(4)[b], 9)) for a in range(4) for b in range(4) if a != b}
check("Remark 4.2: the 12 vertices of K map exactly onto the A_3 roots (e_a - e_b)/2",
      len(verts) == 12 and images == roots)

[PASS] Lemma 4.1: ||W_rho - W_sigma||_1 = max(||v||_inf, ||v||_1/2) (20,000 random v)
[PASS] Lemma 4.1 with actual single-qubit Wigner functions (500 pairs)
[PASS] Remark 4.2: the 12 vertices of K map exactly onto the A_3 roots (e_a - e_b)/2


## 3. Single-qubit maximum (Theorem 4.3)

In [5]:
c0 = (3**.5 - 1) / 2
faces = [np.array(s) / 3**.5 for s in itertools.product([1, -1], repeat=3)]
Cf = [M1.C(bloch(f)) for f in faces]
check("C = (√3-1)/2 at all eight face states", all(close(c, c0, 1e-9) for c in Cf), f"values {np.round(Cf, 9)}")
best = max(M1.C(rand_pure(2)) for _ in range(3000))
check("no random pure single-qubit state exceeds (√3-1)/2 (3,000 samples)", best <= c0 + 1e-9, f"max {best:.9f}")
# uniqueness: states near a face but not at it are strictly below
near = []
for f in faces:
    for _ in range(5):
        g = f + 0.05 * rng.normal(size=3); near.append(M1.C(bloch(g / np.linalg.norm(g))))
check("pure states near (not at) the faces are strictly below the maximum", max(near) < c0 - 1e-6, f"max {max(near):.6f}")
rf = bloch(np.array([1, 1, -1]) / 3**.5)
Gf = M1.Gamma(rf)
check("Gamma(rho_f) = √3 and kappa(rho_f) = 2", close(Gf, 3**.5, 1e-8) and close((Gf - 1) / M1.C(rf), 2, 1e-8))
rT = bloch(np.array([1, 1, 0]) / 2**.5)
check("C(T state) = (√2-1)/2 ≈ 0.2071", close(M1.C(rT), (2**.5 - 1) / 2, 1e-9), f"{M1.C(rT):.7f}")
lam = np.linalg.eigvalsh(A1q(0, 0)).min()
check("(√3-1)/2 = -lambda_min(A_{+++})", close(-lam, c0, 1e-12))

[PASS] C = (√3-1)/2 at all eight face states  —  values [0.366025 0.366025 0.366025 0.366025 0.366025 0.366025 0.366025 0.366025]


[PASS] no random pure single-qubit state exceeds (√3-1)/2 (3,000 samples)  —  max 0.365991097
[PASS] pure states near (not at) the faces are strictly below the maximum  —  max 0.365868
[PASS] Gamma(rho_f) = √3 and kappa(rho_f) = 2
[PASS] C(T state) = (√2-1)/2 ≈ 0.2071  —  0.2071068
[PASS] (√3-1)/2 = -lambda_min(A_{+++})


## 4. Products of single-qubit states (Proposition 4.7, Proposition 4.8, Corollary 4.11)

In [6]:
rg = bloch(np.array([1, 1, 1]) / 3**.5)
Cff = M2.C(np.kron(rf, rf))
check("C(rho_f ⊗ rho_f) = √3/2", close(Cff, 3**.5 / 2, 1e-8), f"{Cff:.9f}")
check("rho_f ⊗ rho_f has 6 negative Wigner entries", int((W2(np.kron(rf, rf)) < -1e-12).sum()) == 6)
check("C(rho_g ⊗ rho_g) = √3/4", close(M2.C(np.kron(rg, rg)), 3**.5 / 4, 1e-8))
check("C(rho_g ⊗ rho_f) = C(rho_f ⊗ rho_g) = 0.6160254",
      close(M2.C(np.kron(rg, rf)), 0.6160254, 1e-6) and close(M2.C(np.kron(rf, rg)), 0.6160254, 1e-6))
viol = 0
for _ in range(300):
    a, b = rand_mixed(2, rng.integers(1, 3)), rand_mixed(2, rng.integers(1, 3))
    if M2.C(np.kron(a, b)) > (1 + M1.C(a)) * (1 + M1.C(b)) - 1 + 1e-8: viol += 1
check("Prop 4.7: C(rho⊗sigma) <= (1+C(rho))(1+C(sigma)) - 1 on 300 random pairs", viol == 0, f"{viol} violations")
C3 = MAG[3].C(kron_all([rf, rf, rf]))
check("Cor 4.11: C(rho_f^⊗3) = ((1+√3)/2)^3 - 1 = 1.5490381", close(C3, ((1 + 3**.5) / 2)**3 - 1, 1e-7), f"{C3:.7f}")
if RUN_SLOW:
    def neg_C_pure2(x):
        v = x[:4] + 1j * x[4:]; return -M2.C(proj(v))
    vals = []
    for _ in range(12):
        r = minimize(neg_C_pure2, rng.normal(size=8), method='Nelder-Mead',
                     options={'maxiter': 2500, 'xatol': 1e-8, 'fatol': 1e-10})
        vals.append(-r.fun)
    check("open problem (v): multistart over two-qubit pure states finds nothing above √3/2",
          max(vals) <= 3**.5 / 2 + 1e-7, f"best found {max(vals):.7f} vs √3/2 = {3**.5/2:.7f}")

[PASS] C(rho_f ⊗ rho_f) = √3/2  —  0.866025404
[PASS] rho_f ⊗ rho_f has 6 negative Wigner entries
[PASS] C(rho_g ⊗ rho_g) = √3/4
[PASS] C(rho_g ⊗ rho_f) = C(rho_f ⊗ rho_g) = 0.6160254


[PASS] Prop 4.7: C(rho⊗sigma) <= (1+C(rho))(1+C(sigma)) - 1 on 300 random pairs  —  0 violations
[PASS] Cor 4.11: C(rho_f^⊗3) = ((1+√3)/2)^3 - 1 = 1.5490381  —  1.5490381


[PASS] open problem (v): multistart over two-qubit pure states finds nothing above √3/2  —  best found 0.8660254 vs √3/2 = 0.8660254


## 5. The tetrahedral dichotomy (Observation 5.1, Table 1, Conjecture 5.5)
The equality branch is numerical in the paper; this section reproduces the evidence.

In [7]:
def sgn_s(r):
    p = np.prod(r); return 0 if abs(p) < 1e-12 else int(np.sign(p))
def rand_bloch_pure():
    v = rng.normal(size=3); return v / np.linalg.norm(v)
eq_neg, tot_neg, eq_other, tot_other, defs = 0, 0, 0, 0, []
for _ in range(200):
    u, v = rand_bloch_pure(), rand_bloch_pure()
    cu, cv = M1.C(bloch(u)), M1.C(bloch(v))
    if cu < 1e-6 or cv < 1e-6: continue
    deficit = (1 + cu) * (1 + cv) - 1 - M2.C(np.kron(bloch(u), bloch(v)))
    if sgn_s(u) <= 0 and sgn_s(v) <= 0:
        tot_neg += 1; eq_neg += deficit < 1e-6
    else:
        tot_other += 1; eq_other += deficit < 1e-6; defs.append(deficit)
check("Obs 5.1: equality in every sampled pair with s(rho), s(sigma) <= 0", eq_neg == tot_neg, f"{eq_neg}/{tot_neg}")
check("Obs 5.1: strict deficit in every other sampled pair", eq_other == 0, f"{eq_other}/{tot_other} equal; min deficit {min(defs):.2e}")

phi = 1.05
cT = M1.C(rT)
table1 = [(np.pi/2, 0.427, 0.000), (1.20, 0.454, 0.136), (0.35, 0.332, 0.121),
          (np.pi - 1.20, 0.590, 0.000), (np.pi - 0.35, 0.453, 0.000)]   # rows 4, 5: theta = 1.94, 2.79
ok = True
for th, val, dlt in table1:
    sig = bloch([np.sin(th)*np.cos(phi), np.sin(th)*np.sin(phi), np.cos(th)])
    c = M2.C(np.kron(rT, sig)); delta = (1 + cT) * (1 + M1.C(sig)) - 1 - c
    ok &= abs(c - val) < 6e-4 and abs(delta - dlt) < 6e-4
    print(f"   theta = {th:.2f}: <Z> = {np.cos(th):+.3f}, C(rho_T ⊗ sigma) = {c:.4f} (paper {val}), delta = {delta:.4f} (paper {dlt})")
check("Table 1 (v4 values) reproduced to 3 decimals, including delta", ok)

# Text after Table 1 and Conjecture 5.5: at fixed sigma the deficit is linear in C(rho) for
# intermediate latitudes, but the slope is NOT proportional to <Z>_sigma. (v3 claimed
# deficit ≈ 0.335 C(rho)<Z>_sigma with R^2 = 0.977; that does not reproduce and was removed.)
slopes = []
for th in (0.45, 0.81, 1.17):
    sig = bloch([np.sin(th)*np.cos(phi), np.sin(th)*np.sin(phi), np.cos(th)]); cs = M1.C(sig)
    xs, ys = [], []
    for a in np.linspace(0.15, np.pi/2 - 0.15, 7):
        rho = bloch([np.cos(a), np.sin(a), 0]); cr = M1.C(rho)
        xs.append(cr); ys.append((1 + cr) * (1 + cs) - 1 - M2.C(np.kron(rho, sig)))
    xs, ys = np.array(xs), np.array(ys); sl = (xs @ ys) / (xs @ xs)
    r2 = 1 - ((ys - sl*xs)**2).sum() / ((ys - ys.mean())**2).sum()
    slopes.append((np.cos(th), sl, r2))
    print(f"   sigma at theta = {th:.2f} (<Z> = {np.cos(th):.3f}): deficit ≈ {sl:.3f} C(rho), R^2 = {r2:.4f}, slope/<Z> = {sl/np.cos(th):.3f}")
check("at fixed sigma the deficit is linear in C(rho) (R^2 > 0.999)", all(r2 > 0.999 for _, _, r2 in slopes))
ratios = [sl / z for z, sl, _ in slopes]
check("the slope is not proportional to <Z>_sigma (so the v3 '0.335 <Z>' law is false)", max(ratios) / min(ratios) > 1.5,
      f"slope/<Z> ranges {min(ratios):.3f}–{max(ratios):.3f}")

[PASS] Obs 5.1: equality in every sampled pair with s(rho), s(sigma) <= 0  —  56/56
[PASS] Obs 5.1: strict deficit in every other sampled pair  —  0/144 equal; min deficit 4.21e-03
   theta = 1.57: <Z> = +0.000, C(rho_T ⊗ sigma) = 0.4274 (paper 0.427), delta = 0.0000 (paper 0.0)
   theta = 1.20: <Z> = +0.362, C(rho_T ⊗ sigma) = 0.4537 (paper 0.454), delta = 0.1364 (paper 0.136)
   theta = 0.35: <Z> = +0.939, C(rho_T ⊗ sigma) = 0.3324 (paper 0.332), delta = 0.1206 (paper 0.121)
   theta = 1.94: <Z> = -0.362, C(rho_T ⊗ sigma) = 0.5901 (paper 0.59), delta = -0.0000 (paper 0.0)
   theta = 2.79: <Z> = -0.939, C(rho_T ⊗ sigma) = 0.4530 (paper 0.453), delta = -0.0000 (paper 0.0)
[PASS] Table 1 (v4 values) reproduced to 3 decimals, including delta
   sigma at theta = 0.45 (<Z> = 0.900): deficit ≈ 0.608 C(rho), R^2 = 1.0000, slope/<Z> = 0.675
   sigma at theta = 0.81 (<Z> = 0.689): deficit ≈ 0.670 C(rho), R^2 = 1.0000, slope/<Z> = 0.971
   sigma at theta = 1.17 (<Z> = 0.390): deficit ≈ 0.662 C(

## 6. The bound $\Gamma \ge 1 + C/M_n$ (Theorem 3.4)

In [8]:
worst = np.inf
for _ in range(200):
    for n, Mg in ((1, M1), (2, M2)):
        rho = rand_pure(2**n) if rng.random() < .5 else rand_mixed(2**n)
        c = Mg.C(rho)
        if c > 1e-6:
            worst = min(worst, Mg.Gamma(rho) - 1 - c / Mg.M_n())
check("Gamma >= 1 + C/M_n on 400 random one- and two-qubit states", worst >= -1e-8, f"min slack {worst:.2e}")

[PASS] Gamma >= 1 + C/M_n on 400 random one- and two-qubit states  —  min slack 1.32e-03


## 7. The three $[[2,1,1]]$ families (Theorems 6.2, 6.5, 8.1; Table 2; Remark 6.3)

In [9]:
k00, k11 = np.eye(4)[0], np.eye(4)[3]
def fam(name, th, n=2):
    z, o = np.zeros(2**n, dtype=complex), np.zeros(2**n, dtype=complex); z[0] = 1; o[-1] = 1
    if name == 'Ry':  return proj(np.cos(th/2) * z + np.sin(th/2) * o)
    if name == 'Rx':  return proj(np.cos(th/2) * z - 1j * np.sin(th/2) * o)
    if name == 'BRz': return proj(z + np.exp(1j * th) * o)
XL, YL, ZL = np.kron(X, X), np.kron(Y, X), np.kron(Z, I2)
thetas = [0.3, 0.7, np.pi/3, 1.3, 1.9, 2.4, 2.9]
okC = okG = True
for th in thetas:
    s, c = abs(np.sin(th)), abs(np.cos(th))
    for name, Cexp in (('Ry', s + c - 1), ('Rx', (s + c - 1) / 2)):
        rho = fam(name, th)
        okC &= close(M2.C(rho), Cexp, 1e-7); okG &= close(M2.Gamma(rho), s + c, 1e-7)
check("Thm 6.2: C(Ry) = |s|+|c|-1, C(Rx) = (|s|+|c|-1)/2 on a theta grid", okC)
check("Thm 6.2: Gamma(Ry) = Gamma(Rx) = |s|+|c|  =>  kappa(Ry) = 1, kappa(Rx) = 2", okG)
okB = True
for th in [0.3, 0.7, np.pi/3, 2.0, 3.6, 5.0]:
    s, c = abs(np.sin(th)), abs(np.cos(th)); rho = fam('BRz', th)
    okB &= close(M2.C(rho), s + c - 1, 1e-7) and close(M2.Gamma(rho), s + c, 1e-7)
check("Thm 6.5: Bell+Rz has C = |s|+|c|-1, Gamma = |s|+|c|, kappa = 1", okB)

def witness_gap(Hop, rho, Mg=M2):
    """Tr(H rho) - max over stabilizer states of Tr(H sigma)."""
    vals = [np.vdot(p, Hop @ p).real for p in Mg.psi]
    return np.trace(Hop @ rho).real - max(vals)
okW = True
for th in [0.3, 0.7, np.pi/3, 1.3]:
    okW &= close(witness_gap(ZL + XL, fam('Ry', th)), M2.C(fam('Ry', th)), 1e-8)
    HRx = 0.5 * (np.sign(np.cos(th)) * ZL - np.sign(np.sin(th)) * YL)
    okW &= close(witness_gap(HRx, fam('Rx', th)), M2.C(fam('Rx', th)), 1e-8)
for th in [0.3, 2.0, 3.6, 5.0]:
    HB = np.sign(np.cos(th)) * XL + np.sign(np.sin(th)) * YL
    okW &= close(witness_gap(HB, fam('BRz', th)), M2.C(fam('BRz', th)), 1e-8)
check("Thm 8.1: the logical Pauli witnesses attain C for all three families", okW)
ZZ = np.kron(Z, Z)
check("Thm 8.1: each witness commutes with the stabilizer ZZ",
      all(np.allclose(Hh @ ZZ, ZZ @ Hh) for Hh in (ZL + XL, ZL - YL, XL + YL)))

th = np.pi/3
negs = {nm: int((W2(fam(nm, th)) < -1e-12).sum()) for nm in ('Ry', 'Rx', 'BRz')}
check("Section 6.3/Table 2: negative Wigner entries at theta = π/3 are 4 (Ry), 2 (Rx), 4 (Bell+Rz)",
      negs == {'Ry': 4, 'Rx': 2, 'BRz': 4}, str(negs))
SI = np.kron(Sg, I2)
check("Remark 6.3: rho_Rx = (S†⊗I) rho_Ry (S⊗I)",
      all(np.allclose(fam('Rx', t), SI.conj().T @ fam('Ry', t) @ SI) for t in thetas))
Ldet = M2.C(fam('BRz', th)); bl = [np.trace(P @ fam('BRz', th)).real for P in (XL, YL)]
check("Thm 8.1: for Bell+Rz, C = |<X_L>| + |<Y_L>| - 1", close(Ldet, abs(bl[0]) + abs(bl[1]) - 1, 1e-8))

[PASS] Thm 6.2: C(Ry) = |s|+|c|-1, C(Rx) = (|s|+|c|-1)/2 on a theta grid
[PASS] Thm 6.2: Gamma(Ry) = Gamma(Rx) = |s|+|c|  =>  kappa(Ry) = 1, kappa(Rx) = 2
[PASS] Thm 6.5: Bell+Rz has C = |s|+|c|-1, Gamma = |s|+|c|, kappa = 1
[PASS] Thm 8.1: the logical Pauli witnesses attain C for all three families
[PASS] Thm 8.1: each witness commutes with the stabilizer ZZ
[PASS] Section 6.3/Table 2: negative Wigner entries at theta = π/3 are 4 (Ry), 2 (Rx), 4 (Bell+Rz)  —  {'Ry': 4, 'Rx': 2, 'BRz': 4}
[PASS] Remark 6.3: rho_Rx = (S†⊗I) rho_Ry (S⊗I)
[PASS] Thm 8.1: for Bell+Rz, C = |<X_L>| + |<Y_L>| - 1


## 8. Depolarizing noise (Section 6.7, Fig. 6)

In [10]:
def depol(rho, p): return (1 - p) * rho + p * np.eye(len(rho)) / len(rho)
for th in (np.pi/3, np.pi/4):
    sc = abs(np.sin(th)) + abs(np.cos(th)); pc = (sc - 1) / (sc - 0.5)
    ps = np.linspace(0.01, 0.98 * pc, 12)
    G = {nm: np.array([M2.Gamma(depol(fam(nm, th), p)) for p in ps]) for nm in ('Rx', 'Ry')}
    Cv = {nm: np.array([M2.C(depol(fam(nm, th), p)) for p in ps]) for nm in ('Rx', 'Ry')}
    kRx, kRy = (G['Rx'] - 1) / Cv['Rx'], (G['Ry'] - 1) / Cv['Ry']
    print(f"   theta = {th:.3f}: p_c = {pc:.4f}")
    print("     kappa(Rx):", np.round(kRx, 4)); print("     kappa(Ry):", np.round(kRy, 4))
    check(f"Section 6.7: Gamma - 1 = (|s|+|c|-1) - p(|s|+|c|-1/2) for both families (theta = {th:.3f})",
          np.allclose(G['Rx'] - 1, (sc - 1) - ps * (sc - .5), atol=1e-7) and np.allclose(G['Ry'], G['Rx'], atol=1e-7))
    check(f"Section 6.7: kappa(Rx) = 2 for all p < p_c (theta = {th:.3f})", np.allclose(kRx, 2, atol=1e-6))
    check(f"Section 6.7: kappa(Ry) rises from 1 to 3/2 and reaches it before p_c (theta = {th:.3f})",
          abs(kRy[0] - 1) < 0.02 and np.all(np.diff(kRy) > -1e-7) and abs(kRy[-1] - 1.5) < 1e-6 and abs(kRy[len(kRy)//2] - 1.5) < 1e-6)
    check(f"Section 6.7: C -> 0 exactly at p_c (theta = {th:.3f})",
          M2.C(depol(fam('Rx', th), pc * 0.999)) > 1e-6 and M2.C(depol(fam('Rx', th), pc * 1.001)) < 1e-9 and
          M2.C(depol(fam('Ry', th), pc * 1.001)) < 1e-9)
    old_pstar = 1 - 1 / sc
    check(f"the v3 threshold 1 - 1/(|s|+|c|) is not where C vanishes (theta = {th:.3f})",
          M2.C(depol(fam('Rx', th), old_pstar)) > 1e-3, f"C there = {M2.C(depol(fam('Rx', th), old_pstar)):.4f}")

   theta = 1.047: p_c = 0.4226
     kappa(Rx): [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
     kappa(Ry): [1.0142 1.0773 1.1657 1.2979 1.5    1.5    1.5    1.5    1.5    1.5    1.5    1.5   ]
[PASS] Section 6.7: Gamma - 1 = (|s|+|c|-1) - p(|s|+|c|-1/2) for both families (theta = 1.047)
[PASS] Section 6.7: kappa(Rx) = 2 for all p < p_c (theta = 1.047)
[PASS] Section 6.7: kappa(Ry) rises from 1 to 3/2 and reaches it before p_c (theta = 1.047)
[PASS] Section 6.7: C -> 0 exactly at p_c (theta = 1.047)
[PASS] the v3 threshold 1 - 1/(|s|+|c|) is not where C vanishes (theta = 1.047)  —  C there = 0.0670
   theta = 0.785: p_c = 0.4531
     kappa(Rx): [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
     kappa(Ry): [1.0125 1.0718 1.1541 1.2759 1.4744 1.5    1.5    1.5    1.5    1.5    1.5    1.5   ]
[PASS] Section 6.7: Gamma - 1 = (|s|+|c|-1) - p(|s|+|c|-1/2) for both families (theta = 0.785)
[PASS] Section 6.7: kappa(Rx) = 2 for all p < p_c (theta = 0.785)
[PASS] Section 6.7: kappa(Ry) rises from 1 to 3/2 and

## 9. Non-monotonicity (Remark 7.1)

In [11]:
HI = np.kron(Hd, I2); inc = []
for _ in range(2000):
    rho = rand_pure(4); inc.append(M2.C(HI @ rho @ HI.conj().T) - M2.C(rho))
inc = np.array(inc); frac = (inc > 1e-9).mean()
print(f"   H⊗I increases C for {100*frac:.1f}% of 2000 Haar-random pure states; largest increase {inc.max():.3f}")
check("Remark 7.1: H⊗I increases C for about 39% of Haar-random pure states, by up to about 0.35",
      0.35 < frac < 0.43 and 0.30 < inc.max() < 0.42, f"{100*frac:.1f}%, max {inc.max():.3f}")
check("Remark 7.1: S⊗I maps Ry onto Rx and halves C",
      close(M2.C(fam('Rx', np.pi/3)), M2.C(fam('Ry', np.pi/3)) / 2, 1e-8))

   H⊗I increases C for 39.1% of 2000 Haar-random pure states; largest increase 0.369
[PASS] Remark 7.1: H⊗I increases C for about 39% of Haar-random pure states, by up to about 0.35  —  39.1%, max 0.369
[PASS] Remark 7.1: S⊗I maps Ry onto Rx and halves C


## 10. $[[n,1,1]]$ repetition code and GHZ families (Prop. 8.4, Prop. 8.5, Obs. 8.6, open-problems table, Remark 8.3)

In [12]:
# Proposition 8.4 (phase parity): <0^n|A_alpha|1^n> = (-1)^P 2^{-n/2} e^{i pi (2m-n)/4}
okP = True
for n in range(1, 7):
    for qs in itertools.product((0, 1), repeat=n):
        for ps in itertools.product((0, 1), repeat=n):
            val = np.prod([A1q(q, p)[0, 1] for q, p in zip(qs, ps)])
            m, P = sum(qs), sum(ps)
            okP &= np.isclose(val, (-1)**P * 2**(-n/2) * np.exp(1j*np.pi*(2*m - n)/4))
check("Prop 8.4: phase-parity formula for the off-diagonal Wigner element (n = 1..6)", okP)

def logical_ops(n):
    return (kron_all([X]*n), kron_all([Y] + [X]*(n-1)), kron_all([Z] + [I2]*(n-1)))

ns = [2, 3] + ([4] if RUN_N4 else [])
for n in ns:
    Mg = MAG[n]; XLn, YLn, ZLn = logical_ops(n)
    for ph in (np.pi/6, np.pi/3):
        Hn = np.sign(np.cos(ph)) * XLn + np.sign(np.sin(ph)) * YLn
        rho = fam('BRz', ph, n); s, c = abs(np.sin(ph)), abs(np.cos(ph))
        maxstab = max(np.vdot(p, Hn @ p).real for p in Mg.psi)
        check(f"Prop 8.5 (n={n}, phi={ph:.3f}): Tr(H*rho) = |s|+|c| and max over STAB = 1",
              close(np.trace(Hn @ rho).real, s + c, 1e-9) and close(maxstab, 1, 1e-9))
        t = time.time(); Cn = Mg.C(rho); Gn = Mg.Gamma(rho)
        target = (s + c - 1) * (2 if n == 4 else 1)
        label = "Obs 8.6: C doubles at n=4" if n == 4 else "Prop 8.5: C = |s|+|c|-1"
        check(f"{label} (n={n}, phi={ph:.3f})", close(Cn, target, 1e-7), f"C = {Cn:.7f}, target {target:.7f} ({time.time()-t:.0f}s)")
        check(f"Obs 8.6: Gamma(GHZ+Rz) = |s|+|c| (n={n}, phi={ph:.3f})", close(Gn, s + c, 1e-7), f"{Gn:.7f}")

# n-qubit Ry family in the (cos t, sin t) convention: C = |cos 2t| + |sin 2t| - 1 for n in {2,3}
okR = True
for n in (2, 3):
    for t in (0.3, 0.5, 1.1):
        z = np.zeros(2**n); z[0] = 1; o = np.zeros(2**n); o[-1] = 1
        okR &= close(MAG[n].C(proj(np.cos(t)*z + np.sin(t)*o)), abs(np.cos(2*t)) + abs(np.sin(2*t)) - 1, 1e-7)
check("Section 8.3: n-qubit Ry family has C = |cos 2t| + |sin 2t| - 1 for n = 2, 3", okR)

# open-problems table: kappa for Ry, Rx, Bell+Rz at n = 2, 3 (, 4)
expected = {2: {'Ry': 1, 'Rx': 2, 'BRz': 1}, 3: {'Ry': 1, 'Rx': 1, 'BRz': 1}, 4: {'Ry': 1, 'Rx': 1, 'BRz': .5}}
for n in ns:
    Mg = MAG[n]; got = {}
    for nm in ('Ry', 'Rx', 'BRz'):
        vals = []
        for th in (np.pi/3, 0.7):
            rho = fam(nm, th, n); vals.append((Mg.Gamma(rho) - 1) / Mg.C(rho))
        got[nm] = vals
    ok = all(np.allclose(got[nm], expected[n][nm], atol=1e-6) for nm in got)
    check(f"open-problems table, n = {n}: kappa(Ry, Rx, Bell+Rz) = {tuple(expected[n].values())}", ok,
          str({k: np.round(v, 6).tolist() for k, v in got.items()}))

# Remark 8.3: Z_L + X_L stays optimal for Ry; is the LP's own optimal witness logical?
for n in ns:
    Mg = MAG[n]; XLn, YLn, ZLn = logical_ops(n); rho = fam('Ry', np.pi/3, n)
    check(f"Remark 8.3 (n={n}): Z_L + X_L is an optimal witness for Ry",
          close(witness_gap(ZLn + XLn, rho, Mg), Mg.C(rho), 1e-7))
    Cv, y = Mg.C(rho, witness=True)
    if y is not None:
        Hy = sum(yi * A for yi, A in zip(y, Mg.Aops)) / Mg.dim
        gens = [kron_all([I2]*i + [Z, Z] + [I2]*(n-i-2)) for i in range(n-1)]
        comm = all(np.allclose(Hy @ g, g @ Hy, atol=1e-8) for g in gens)
        if n == 2:
            check("Remark 8.3 (n=2): the LP's optimal witness is logical (commutes with ZZ)", comm)
        else:
            check(f"Remark 8.3 (n={n}): the LP also returns a non-logical optimal witness (non-uniqueness)", not comm)

[PASS] Prop 8.4: phase-parity formula for the off-diagonal Wigner element (n = 1..6)
[PASS] Prop 8.5 (n=2, phi=0.524): Tr(H*rho) = |s|+|c| and max over STAB = 1
[PASS] Prop 8.5: C = |s|+|c|-1 (n=2, phi=0.524)  —  C = 0.3660254, target 0.3660254 (0s)
[PASS] Obs 8.6: Gamma(GHZ+Rz) = |s|+|c| (n=2, phi=0.524)  —  1.3660254
[PASS] Prop 8.5 (n=2, phi=1.047): Tr(H*rho) = |s|+|c| and max over STAB = 1
[PASS] Prop 8.5: C = |s|+|c|-1 (n=2, phi=1.047)  —  C = 0.3660254, target 0.3660254 (0s)
[PASS] Obs 8.6: Gamma(GHZ+Rz) = |s|+|c| (n=2, phi=1.047)  —  1.3660254
[PASS] Prop 8.5 (n=3, phi=0.524): Tr(H*rho) = |s|+|c| and max over STAB = 1
[PASS] Prop 8.5: C = |s|+|c|-1 (n=3, phi=0.524)  —  C = 0.3660254, target 0.3660254 (0s)
[PASS] Obs 8.6: Gamma(GHZ+Rz) = |s|+|c| (n=3, phi=0.524)  —  1.3660254
[PASS] Prop 8.5 (n=3, phi=1.047): Tr(H*rho) = |s|+|c| and max over STAB = 1


[PASS] Prop 8.5: C = |s|+|c|-1 (n=3, phi=1.047)  —  C = 0.3660254, target 0.3660254 (0s)
[PASS] Obs 8.6: Gamma(GHZ+Rz) = |s|+|c| (n=3, phi=1.047)  —  1.3660254
[PASS] Prop 8.5 (n=4, phi=0.524): Tr(H*rho) = |s|+|c| and max over STAB = 1


[PASS] Obs 8.6: C doubles at n=4 (n=4, phi=0.524)  —  C = 0.7320508, target 0.7320508 (41s)
[PASS] Obs 8.6: Gamma(GHZ+Rz) = |s|+|c| (n=4, phi=0.524)  —  1.3660254
[PASS] Prop 8.5 (n=4, phi=1.047): Tr(H*rho) = |s|+|c| and max over STAB = 1


[PASS] Obs 8.6: C doubles at n=4 (n=4, phi=1.047)  —  C = 0.7320508, target 0.7320508 (33s)
[PASS] Obs 8.6: Gamma(GHZ+Rz) = |s|+|c| (n=4, phi=1.047)  —  1.3660254
[PASS] Section 8.3: n-qubit Ry family has C = |cos 2t| + |sin 2t| - 1 for n = 2, 3
[PASS] open-problems table, n = 2: kappa(Ry, Rx, Bell+Rz) = (1, 2, 1)  —  {'Ry': [1.0, 1.0], 'Rx': [2.0, 2.0], 'BRz': [1.0, 1.0]}


[PASS] open-problems table, n = 3: kappa(Ry, Rx, Bell+Rz) = (1, 1, 1)  —  {'Ry': [1.0, 1.0], 'Rx': [1.0, 1.0], 'BRz': [1.0, 1.0]}


[PASS] open-problems table, n = 4: kappa(Ry, Rx, Bell+Rz) = (1, 1, 0.5)  —  {'Ry': [1.0, 1.0], 'Rx': [1.0, 1.0], 'BRz': [0.5, 0.5]}
[PASS] Remark 8.3 (n=2): Z_L + X_L is an optimal witness for Ry
[PASS] Remark 8.3 (n=2): the LP's optimal witness is logical (commutes with ZZ)
[PASS] Remark 8.3 (n=3): Z_L + X_L is an optimal witness for Ry
[PASS] Remark 8.3 (n=3): the LP also returns a non-logical optimal witness (non-uniqueness)


[PASS] Remark 8.3 (n=4): Z_L + X_L is an optimal witness for Ry


[PASS] Remark 8.3 (n=4): the LP also returns a non-logical optimal witness (non-uniqueness)


## 11. Frame freedom (Remark 4.12)
### 11.1 One qubit: eight frames, identical C on every state

In [13]:
frames1 = list(itertools.product([1, -1], repeat=3))
ok = True
for a in frames1:
    F = [A1q(q, p, a) for q in (0, 1) for p in (0, 1)]
    G = np.array([[np.trace(x @ y).real for y in F] for x in F])
    ok &= np.allclose(G, 2 * np.eye(4)) and np.allclose(sum(F), 2 * I2)
check("all eight a ∈ {±1}^3 give valid frames", ok)
# completeness: |a|^2 = 3 and orthogonality to the X-translate force a_i^2 = 1 (checked on a grid)
MF = [Magic(1, Aops=[A1q(q, p, a) for q in (0, 1) for p in (0, 1)], states=M1.psi) for a in frames1]
tests = [rand_mixed(2, rng.integers(1, 3)) for _ in range(200)]
spread = max(np.ptp([Mf.C(r) for Mf in MF]) for r in tests)
check("C is identical in all eight single-qubit frames (200 random states)", spread < 1e-8, f"max spread {spread:.1e}")

[PASS] all eight a ∈ {±1}^3 give valid frames


[PASS] C is identical in all eight single-qubit frames (200 random states)  —  max spread 7.8e-16


### 11.2 Two qubits: the 2048 Pauli-covariant frames

In [14]:
mats = [np.kron(P1[i], P1[j]) for i in range(4) for j in range(4) if (i, j) != (0, 0)]
Dsp = [np.kron(a, b) for a in P1 for b in P1]
chi = np.array([[1 if np.allclose(d @ Mm, Mm @ d) else -1 for Mm in mats] for d in Dsp])
Sv = np.array([[np.vdot(p, Mm @ p).real for Mm in mats] for p in M2.psi])
def Wmat_eps(e): return (1 + (chi * e) @ Sv.T) / 16                 # 16 x 60
def C_eps(e, pv, Ws=None):
    Ws = Wmat_eps(e) if Ws is None else Ws; w0 = (1 + (chi * e) @ pv) / 16
    r = linprog(np.r_[np.zeros(60), np.ones(16)],
                A_ub=np.block([[-Ws, -np.eye(16)], [Ws, -np.eye(16)]]), b_ub=np.r_[-w0, w0],
                A_eq=np.r_[np.ones(60), np.zeros(16)][None], b_eq=[1], bounds=(0, None), method='highs')
    return r.fun
pvec = lambda r: np.array([np.trace(r @ Mm).real for Mm in mats])

allp = list(itertools.product([1, -1], repeat=15)); nvalid = 0
for e in allp:
    A0 = 0.25 * (np.eye(4) + sum(s * Mm for s, Mm in zip(e, mats)))
    Ao = np.array([d @ A0 @ d.conj().T for d in Dsp])
    G = np.einsum('aij,bji->ab', Ao, Ao).real
    nvalid += np.allclose(Ao.sum(0), 4 * np.eye(4)) and np.allclose(G, 4 * np.eye(16))
check("all 2^15 = 32768 sign patterns give valid Pauli-covariant frames", nvalid == 32768)
rep, reps = {}, []
for e in allp:
    if e in rep: continue
    orb = [tuple(np.array(e) * chi[a]) for a in range(16)]; r0 = min(orb)
    for x in orb: rep[x] = r0
    reps.append(r0)
check("up to translation there are 2048 distinct frames", len(reps) == 2048)

def plab(a, b):
    Mx = mats[a] @ mats[b]
    for l, Q in enumerate(mats):
        for ph in (1, -1, 1j, -1j):
            if np.allclose(Mx, ph * Q): return l, ph
trip = {}
for a, b in itertools.combinations(range(15), 2):
    if np.allclose(mats[a] @ mats[b], mats[b] @ mats[a]):
        c, ph = plab(a, b); trip[tuple(sorted((a, b, c)))] = int(np.real(ph))
parts = []
def srch(used, ch):
    if len(ch) == 5: parts.append(ch); return
    f = min(set(range(15)) - used)
    for t in trip:
        if f in t and not set(t) & used: srch(used | set(t), ch + [t])
srch(set(), [])
mub = set()
for part in parts:
    for sg in itertools.product([1, -1], repeat=10):
        e = np.zeros(15, int)
        for k, (a, b, c) in enumerate(part):
            s1, s2 = sg[2*k], sg[2*k+1]; e[a], e[b], e[c] = s1, s2, int(np.real(plab(a, b)[1])) * s1 * s2
        mub.add(tuple(e))
mubreps = {rep[e] for e in mub}
check("triple-partition (MUB) construction: 6 partitions, 4864 origins, 304 frames, product frame included",
      len(parts) == 6 and len(mub) == 4864 and len(mubreps) == 304 and tuple([1]*15) in mub)

def ratio_classes(th):
    rx_, ry_ = pvec(fam('Rx', th)), pvec(fam('Ry', th)); res = {}
    for r0 in reps:
        e = np.array(r0); Ws = Wmat_eps(e); res[r0] = (C_eps(e, rx_, Ws), C_eps(e, ry_, Ws))
    return res
res = ratio_classes(np.pi/3)
cls = collections.Counter((round(a, 6), round(b, 6)) for a, b in res.values())
clsM = collections.Counter((round(res[r][0], 6), round(res[r][1], 6)) for r in mubreps)
print("   (C_Rx, C_Ry) classes, all frames:", dict(cls)); print("   triple-partition frames:", dict(clsM))
check("theta = π/3: (C_Rx, C_Ry) takes each value in {c0/2, c0}^2 on exactly 512 frames",
      sorted(cls.values()) == [512]*4 and {round(c0/2, 6), round(c0, 6)} == {k[0] for k in cls})
check("triple-partition frames split 80/80/80/64 (64 = both values c0/2)",
      sorted(clsM.values()) == [64, 80, 80, 80] and clsM[(round(c0/2, 6), round(c0/2, 6))] == 64)
ok = True
for th in (0.3, 1.0, 2.0, 2.6):
    rr = ratio_classes(th)
    rc = collections.Counter(round(a / b, 5) for a, b in rr.values())
    ok &= rc == collections.Counter({0.5: 512, 1.0: 1024, 2.0: 512})
check("the same ratio classes (1/2: 512, 1: 1024, 2: 512) at theta = 0.3, 1.0, 2.0, 2.6", ok)

# product frames and the phase-gate argument
prodreps = set()
for a in itertools.product([1, -1], repeat=3):
    for b in itertools.product([1, -1], repeat=3):
        A_, B_ = (1,) + a, (1,) + b
        e = tuple(A_[i] * B_[j] for i in range(4) for j in range(4) if (i, j) != (0, 0))
        prodreps.add(rep[e])
ratios_prod = sorted(round(res[r][0] / res[r][1], 6) for r in prodreps)
check("product frames: 4 up to translation, ratio 1/2 in two and 2 in the other two (chirality)",
      len(prodreps) == 4 and ratios_prod == [0.5, 0.5, 2.0, 2.0], str(ratios_prod))

[PASS] all 2^15 = 32768 sign patterns give valid Pauli-covariant frames
[PASS] up to translation there are 2048 distinct frames


[PASS] triple-partition (MUB) construction: 6 partitions, 4864 origins, 304 frames, product frame included


   (C_Rx, C_Ry) classes, all frames: {(0.183013, 0.366025): 512, (0.366025, 0.183013): 512, (0.183013, 0.183013): 512, (0.366025, 0.366025): 512}
   triple-partition frames: {(0.366025, 0.366025): 80, (0.366025, 0.183013): 80, (0.183013, 0.366025): 80, (0.183013, 0.183013): 64}
[PASS] theta = π/3: (C_Rx, C_Ry) takes each value in {c0/2, c0}^2 on exactly 512 frames
[PASS] triple-partition frames split 80/80/80/64 (64 = both values c0/2)


[PASS] the same ratio classes (1/2: 512, 1: 1024, 2: 512) at theta = 0.3, 1.0, 2.0, 2.6
[PASS] product frames: 4 up to translation, ratio 1/2 in two and 2 in the other two (chirality)  —  [0.5, 0.5, 2.0, 2.0]


### 11.3 $M_2 = 2$ in every frame: the Mermin–Peres obstruction

In [15]:
M2all = collections.Counter(round(np.abs(Wmat_eps(np.array(r))).sum(0).max(), 9) for r in reps)
check("M_2 = 2 in all 2048 frames", M2all == collections.Counter({2.0: 2048}), str(dict(M2all)))
ok = True
for r0 in reps:
    e = np.array(r0); nrm = np.abs(Wmat_eps(e)).sum(0)
    for j in range(60):
        sup = tuple(sorted(np.nonzero(np.abs(Sv[j]) > .5)[0]))
        ok &= close(nrm[j], 2 if trip[sup] * np.prod(e[list(sup)]) == -1 else 1, 1e-9)
check("criterion: ||W_sigma||_1 = 2 iff lambda_S c_P1 c_P2 c_P3 = -1 (2048 frames x 60 states)", ok)
square = None
for six in itertools.combinations(trip, 6):
    cnt = collections.Counter(p for t in six for p in t)
    if len(cnt) == 9 and all(v == 2 for v in cnt.values()) and np.prod([trip[t] for t in six]) == -1:
        square = six; break
labels = [a + b for a in 'IXYZ' for b in 'IXYZ'][1:]
print("   Mermin–Peres square (six commuting triples, product of lambda = -1):",
      [[labels[i] for i in t] for t in square])
check("a Mermin–Peres square exists among the 15 commuting triples, so no frame has M_2 = 1", square is not None)

[PASS] M_2 = 2 in all 2048 frames  —  {np.float64(2.0): 2048}


[PASS] criterion: ||W_sigma||_1 = 2 iff lambda_S c_P1 c_P2 c_P3 = -1 (2048 frames x 60 states)
   Mermin–Peres square (six commuting triples, product of lambda = -1): [['IX', 'XI', 'XX'], ['IX', 'YI', 'YX'], ['IY', 'XI', 'XY'], ['IY', 'YI', 'YY'], ['XX', 'YY', 'ZZ'], ['XY', 'YX', 'ZZ']]
[PASS] a Mermin–Peres square exists among the 15 commuting triples, so no frame has M_2 = 1


### 11.4 The separable maximum outside product frames (numerical; `RUN_SLOW`)

In [16]:
if RUN_SLOW:
    fp = [pvec(np.kron(bloch(f), bloch(g))) for f in faces for g in faces]
    t = time.time(); mx = {}
    for r0 in reps:
        e = np.array(r0); Ws = Wmat_eps(e); mx[r0] = max(C_eps(e, p, Ws) for p in fp)
    cnt = collections.Counter(round(v, 6) for v in mx.values())
    print(f"   max over face⊗face products per frame ({time.time()-t:.0f}s):", dict(sorted(cnt.items())))
    check("face⊗face products attain √3/2 in exactly 128 of the 2048 frames", cnt[round(3**.5/2, 6)] == 128)
    check("all four product frames attain √3/2", all(close(mx[r], 3**.5/2, 1e-7) for r in prodreps))
    sv = lambda t_, p_: np.array([np.sin(t_)*np.cos(p_), np.sin(t_)*np.sin(p_), np.cos(t_)])
    groups = collections.defaultdict(list)
    for r0 in reps: groups[round(mx[r0], 6)].append(r0)
    best_outside = 0
    for k, members in sorted(groups.items()):
        if k == round(3**.5/2, 6): continue
        r0 = members[0]; e = np.array(r0); Ws = Wmat_eps(e)
        f = lambda x: -C_eps(e, pvec(np.kron(bloch(sv(x[0], x[1])), bloch(sv(x[2], x[3])))), Ws)
        vals = [-minimize(f, rng.uniform(0, 1, 4) * [np.pi, 2*np.pi, np.pi, 2*np.pi], method='Nelder-Mead',
                          options={'maxiter': 1500, 'xatol': 1e-7, 'fatol': 1e-10}).fun for _ in range(8)]
        best_outside = max(best_outside, max(vals))
        print(f"   class with face-max {k}: multistart separable max {max(vals):.6f}")
    check("multistart finds no product state reaching √3/2 outside those 128 frames (lower bounds only)",
          best_outside < 3**.5/2 - 1e-3, f"best {best_outside:.4f}")

   max over face⊗face products per frame (336s): {0.654701: 768, 0.699359: 1080, 0.715705: 72, 0.866025: 128}
[PASS] face⊗face products attain √3/2 in exactly 128 of the 2048 frames
[PASS] all four product frames attain √3/2


   class with face-max 0.654701: multistart separable max 0.666123


   class with face-max 0.699359: multistart separable max 0.739251


   class with face-max 0.715705: multistart separable max 0.739251
[PASS] multistart finds no product state reaching √3/2 outside those 128 frames (lower bounds only)  —  best 0.7393


## 12. Odd-prime comparison used in Section 5.1 (one qutrit)
The paper states only that $C_d \ge \lVert W_\rho\rVert_1 - 1$ for odd prime $d$, with strict
inequality possible. Checked here for $d=3$ with the Gross/Wootters Wigner function.

In [17]:
d3 = 3; w3 = np.exp(2j*np.pi/3)
X3 = np.roll(np.eye(3), 1, axis=0).astype(complex); Z3 = np.diag([w3**j for j in range(3)])
D3 = lambda q, p: w3**(2*q*p) * np.linalg.matrix_power(X3, q) @ np.linalg.matrix_power(Z3, p)
Par = np.zeros((3, 3), complex)
for j in range(3): Par[(-j) % 3, j] = 1
A3 = [D3(q, p) @ Par @ D3(q, p).conj().T for q in range(3) for p in range(3)]
# the 12 qutrit stabilizer states: eigenbases of Z, X, XZ, XZ^2
st3 = []
for Mx in (Z3, X3, X3 @ Z3, X3 @ Z3 @ Z3):
    _, V = np.linalg.eig(Mx); st3 += [V[:, k] for k in range(3)]
W3 = lambda r: np.array([np.trace(r @ A).real for A in A3]) / 3
WS3 = np.array([W3(proj(v)) for v in st3])
def C3(r):
    w0 = W3(r); res = linprog(np.r_[np.zeros(12), np.ones(9)],
        A_ub=np.block([[-WS3.T, -np.eye(9)], [WS3.T, -np.eye(9)]]), b_ub=np.r_[-w0, w0],
        A_eq=np.r_[np.ones(12), np.zeros(9)][None], b_eq=[1], bounds=(0, None), method='highs')
    return res.fun
check("qutrit: 12 stabilizer states, all Wigner-nonnegative", len(st3) == 12 and WS3.min() > -1e-12)
gaps = [C3(r) - (np.abs(W3(r)).sum() - 1) for r in (rand_pure(3) for _ in range(300))]
check("qutrit: C_3 >= ||W||_1 - 1 on 300 random pure states, strictly for some", min(gaps) > -1e-9 and max(gaps) > 1e-3,
      f"min gap {min(gaps):.1e}, max gap {max(gaps):.4f}")

[PASS] qutrit: 12 stabilizer states, all Wigner-nonnegative


[PASS] qutrit: C_3 >= ||W||_1 - 1 on 300 random pure states, strictly for some  —  min gap -4.4e-16, max gap 0.0279


## 13. Ledger

In [18]:
npass = sum(ok for _, ok, _ in LEDGER)
print(f"{npass} / {len(LEDGER)} checks passed   (RUN_N4={RUN_N4}, RUN_SLOW={RUN_SLOW}, total {time.time()-T0:.0f}s)\n")
for name, ok, detail in LEDGER:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print("\nNot reproduced here: C(Hoggar) ≈ 1.2262 (open problem (v)) — the Hoggar state is defined in Ref. [Palhares2026].")

98 / 98 checks passed   (RUN_N4=True, RUN_SLOW=True, total 882s)

PASS  number of 1-qubit stabilizer states = 6
PASS  number of 2-qubit stabilizer states = 60
PASS  number of 3-qubit stabilizer states = 1080
PASS  number of 4-qubit stabilizer states = 36720
PASS  M_1 = 1
PASS  M_2 = 2 (attained by Bell states)
PASS  eigenvalues of a two-qubit A_alpha are -1/2, -1/2, (2-√3)/2, (2+√3)/2
PASS  Prop C.1(2): -1/8 <= W <= (2+√3)/8 on 20,000 random pure states
PASS  Prop C.1(3): sum W^2 = 1/4 for pure states
PASS  Appendix C: random pure states reach 9 negative entries (the old 'at most 6' claim was false)
PASS  Lemma 4.1: ||W_rho - W_sigma||_1 = max(||v||_inf, ||v||_1/2) (20,000 random v)
PASS  Lemma 4.1 with actual single-qubit Wigner functions (500 pairs)
PASS  Remark 4.2: the 12 vertices of K map exactly onto the A_3 roots (e_a - e_b)/2
PASS  C = (√3-1)/2 at all eight face states
PASS  no random pure single-qubit state exceeds (√3-1)/2 (3,000 samples)
PASS  pure states near (not at) the f